In [ ]:
# ============================================================
# Agentic Text-to-SQL System
# Stack: LangGraph + Groq (Llama) + SQLite + Gradio
# Run in Google Colab
# ============================================================

In [ ]:
#  Install Dependencies ---
!pip install langgraph langchain-groq langchain-core gradio sqlalchemy --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.3 MB/s eta 0:00:00


In [ ]:
# Imports & Setup

import os
import sqlite3
import re
from typing import TypedDict, Annotated, List, Optional
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import gradio as gr

from groq import Groq
from google.colab import userdata


In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
userdata.get('GROQ_API_KEY')
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
print(client)

In [ ]:
# Create Sample Database
DB_PATH = "sample_company.db"

def create_sample_database():
    """Create a sample company database with realistic data."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Drop existing tables
    cursor.executescript("""
        DROP TABLE IF EXISTS order_items;
        DROP TABLE IF EXISTS orders;
        DROP TABLE IF EXISTS products;
        DROP TABLE IF EXISTS customers;
        DROP TABLE IF EXISTS employees;
        DROP TABLE IF EXISTS departments;
    """)

    # Create tables
    cursor.executescript("""
        CREATE TABLE departments (
            department_id INTEGER PRIMARY KEY,
            department_name TEXT NOT NULL,
            location TEXT NOT NULL
        );

        CREATE TABLE employees (
            employee_id INTEGER PRIMARY KEY,
            first_name TEXT NOT NULL,
            last_name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            hire_date DATE NOT NULL,
            salary REAL NOT NULL,
            department_id INTEGER,
            manager_id INTEGER,
            FOREIGN KEY (department_id) REFERENCES departments(department_id),
            FOREIGN KEY (manager_id) REFERENCES employees(employee_id)
        );

        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            city TEXT NOT NULL,
            country TEXT NOT NULL,
            registration_date DATE NOT NULL
        );

        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            product_name TEXT NOT NULL,
            category TEXT NOT NULL,
            price REAL NOT NULL,
            stock_quantity INTEGER NOT NULL
        );

        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL,
            order_date DATE NOT NULL,
            total_amount REAL NOT NULL,
            status TEXT NOT NULL CHECK(status IN ('pending','shipped','delivered','cancelled')),
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        );

        CREATE TABLE order_items (
            item_id INTEGER PRIMARY KEY,
            order_id INTEGER NOT NULL,
            product_id INTEGER NOT NULL,
            quantity INTEGER NOT NULL,
            unit_price REAL NOT NULL,
            FOREIGN KEY (order_id) REFERENCES orders(order_id),
            FOREIGN KEY (product_id) REFERENCES products(product_id)
        );
    """)

    # Insert sample data
    cursor.executescript("""
        INSERT INTO departments VALUES
        (1, 'Engineering', 'San Francisco'),
        (2, 'Marketing', 'New York'),
        (3, 'Sales', 'Chicago'),
        (4, 'HR', 'San Francisco'),
        (5, 'Finance', 'New York');

        INSERT INTO employees VALUES
        (1, 'Alice', 'Johnson', 'alice@company.com', '2020-01-15', 125000, 1, NULL),
        (2, 'Bob', 'Smith', 'bob@company.com', '2020-03-20', 115000, 1, 1),
        (3, 'Carol', 'Williams', 'carol@company.com', '2019-06-10', 95000, 2, NULL),
        (4, 'David', 'Brown', 'david@company.com', '2021-02-01', 88000, 3, NULL),
        (5, 'Eva', 'Davis', 'eva@company.com', '2021-07-15', 92000, 1, 1),
        (6, 'Frank', 'Miller', 'frank@company.com', '2022-01-10', 78000, 2, 3),
        (7, 'Grace', 'Wilson', 'grace@company.com', '2022-05-20', 105000, 4, NULL),
        (8, 'Henry', 'Moore', 'henry@company.com', '2023-03-01', 70000, 3, 4),
        (9, 'Ivy', 'Taylor', 'ivy@company.com', '2023-06-15', 82000, 5, NULL),
        (10, 'Jack', 'Anderson', 'jack@company.com', '2024-01-10', 72000, 1, 2);

        INSERT INTO customers VALUES
        (1, 'TechCorp Inc', 'info@techcorp.com', 'San Francisco', 'USA', '2022-01-10'),
        (2, 'Global Retail', 'contact@globalretail.com', 'London', 'UK', '2022-03-15'),
        (3, 'StartupXYZ', 'hello@startupxyz.com', 'Berlin', 'Germany', '2022-06-20'),
        (4, 'MegaStore', 'sales@megastore.com', 'Tokyo', 'Japan', '2023-01-05'),
        (5, 'CloudNine Ltd', 'support@cloudnine.com', 'Sydney', 'Australia', '2023-04-12'),
        (6, 'DataDriven Co', 'info@datadriven.com', 'New York', 'USA', '2023-07-01'),
        (7, 'InnovateTech', 'team@innovatetech.com', 'Toronto', 'Canada', '2023-09-15'),
        (8, 'SmartSolutions', 'hello@smartsol.com', 'Mumbai', 'India', '2024-01-20');

        INSERT INTO products VALUES
        (1, 'Laptop Pro 15', 'Electronics', 1299.99, 150),
        (2, 'Wireless Mouse', 'Accessories', 29.99, 500),
        (3, 'Mechanical Keyboard', 'Accessories', 89.99, 300),
        (4, 'Monitor 27 inch', 'Electronics', 449.99, 200),
        (5, 'USB-C Hub', 'Accessories', 49.99, 400),
        (6, 'Webcam HD', 'Electronics', 79.99, 250),
        (7, 'Standing Desk', 'Furniture', 599.99, 100),
        (8, 'Ergonomic Chair', 'Furniture', 399.99, 120),
        (9, 'Noise-Cancel Headphones', 'Audio', 249.99, 180),
        (10, 'Portable SSD 1TB', 'Storage', 119.99, 350);

        INSERT INTO orders VALUES
        (1, 1, '2024-01-15', 1579.97, 'delivered'),
        (2, 2, '2024-01-20', 449.99, 'delivered'),
        (3, 3, '2024-02-10', 169.98, 'delivered'),
        (4, 1, '2024-02-28', 849.98, 'shipped'),
        (5, 4, '2024-03-05', 1299.99, 'shipped'),
        (6, 5, '2024-03-15', 599.99, 'pending'),
        (7, 6, '2024-03-20', 329.98, 'delivered'),
        (8, 2, '2024-04-01', 719.98, 'pending'),
        (9, 7, '2024-04-10', 249.99, 'shipped'),
        (10, 8, '2024-04-15', 1949.97, 'pending'),
        (11, 3, '2024-04-20', 89.99, 'cancelled'),
        (12, 1, '2024-05-01', 499.98, 'delivered');

        INSERT INTO order_items VALUES
        (1, 1, 1, 1, 1299.99),
        (2, 1, 2, 2, 29.99),
        (3, 1, 5, 1, 49.99),
        (4, 2, 4, 1, 449.99),
        (5, 3, 2, 1, 29.99),
        (6, 3, 3, 1, 89.99),
        (7, 3, 5, 1, 49.99),
        (8, 4, 4, 1, 449.99),
        (9, 4, 8, 1, 399.99),
        (10, 5, 1, 1, 1299.99),
        (11, 6, 7, 1, 599.99),
        (12, 7, 9, 1, 249.99),
        (13, 7, 6, 1, 79.99),
        (14, 8, 7, 1, 599.99),
        (15, 8, 10, 1, 119.99),
        (16, 9, 9, 1, 249.99),
        (17, 10, 1, 1, 1299.99),
        (18, 10, 7, 1, 599.99),
        (19, 10, 5, 1, 49.99),
        (20, 11, 3, 1, 89.99),
        (21, 12, 9, 2, 249.99);
    """)

    conn.commit()
    conn.close()
    print("Sample database created successfully!")


create_sample_database()


Sample database created successfully!


In [ ]:

# Helper Functions ---
def get_full_schema() -> str:
    """Get the complete database schema."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL;")
    schemas = cursor.fetchall()
    conn.close()
    return "\n\n".join(s[0] for s in schemas)


def get_table_names() -> List[str]:
    """Get all table names from the database."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    conn.close()
    return tables


def get_sample_data(table_name: str, limit: int = 3) -> str:
    """Get sample rows from a table for context."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    # Validate table name against actual tables to prevent SQL injection
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    valid_tables = {row[0] for row in cursor.fetchall()}
    if table_name not in valid_tables:
        conn.close()
        return f"Table '{table_name}' not found."
    cursor.execute(f'SELECT * FROM "{table_name}" LIMIT ?', (limit,))
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    conn.close()
    result = f"Columns: {columns}\n"
    for row in rows:
        result += f"  {row}\n"
    return result


# --- Robust SQL Safety Layer ---
DISALLOWED_KEYWORDS = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|VACUUM|ATTACH|DETACH|PRAGMA|TRUNCATE)\b",
    re.I,
)
_CODE_FENCE = re.compile(r"^\s*```(?:sql)?\s*|\s*```\s*$", re.I)
_LEADING_COMMENTS = re.compile(r"^\s*(?:--[^\n]*\n|/\*.*?\*/\s*)*", re.S)
_LABEL_PREFIX = re.compile(r"^\s*(sql\s*query|query)\s*:\s*", re.I)


def _normalize_sql(query: str) -> str:
    """Strip code fences, comments, labels, and trailing semicolons from LLM output."""
    q = query.strip()
    q = _CODE_FENCE.sub("", q).strip()
    q = _LEADING_COMMENTS.sub("", q).strip()
    q = _LABEL_PREFIX.sub("", q).strip()
    q = q.rstrip(";").strip()
    return q


def execute_sql(sql: str, max_rows: int = 200) -> dict:
    """Execute a SQL query with robust safety checks and return results."""
    q = _normalize_sql(sql)

    # Block multi-statement queries (SQL injection vector)
    if ";" in q:
        return {"success": False, "error": "Multiple SQL statements are not allowed."}

    # Block non-read-only keywords
    if DISALLOWED_KEYWORDS.search(q):
        return {"success": False, "error": "Only read-only SELECT queries are allowed."}

    # Only allow SELECT/WITH
    if not q.lower().startswith(("select", "with")):
        return {"success": False, "error": "Only SELECT/WITH queries are allowed."}

    # Block unsupported SQLite joins
    if re.search(r"\b(full\s+outer\s+join|right\s+join)\b", q, re.I):
        return {"success": False, "error": "SQLite does not support FULL OUTER JOIN / RIGHT JOIN. Use LEFT JOIN instead."}

    # Auto-append LIMIT if missing
    if "limit" not in q.lower():
        q = f"{q} LIMIT {max_rows}"

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    try:
        cursor.execute(q)
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        conn.close()
        return {"success": True, "columns": columns, "rows": rows, "row_count": len(rows)}
    except Exception as e:
        conn.close()
        return {"success": False, "error": str(e)}


def extract_sql(text: str) -> str:
    """Extract and normalize SQL query from LLM response."""
    # Try to find SQL in code blocks
    patterns = [
        r"```sql\s*(.*?)\s*```",
        r"```\s*(SELECT.*?)\s*```",
        r"```\s*(WITH.*?)\s*```",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            return _normalize_sql(match.group(1)) + ";"

    # If no code block, try to find a SELECT/WITH statement
    match = re.search(r"((?:SELECT|WITH)\s+.+?)(?:;|$)", text, re.DOTALL | re.IGNORECASE)
    if match:
        return _normalize_sql(match.group(1)) + ";"

    # Last resort
    return text.strip()


In [ ]:
## LLM Call
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=2048,
    api_key=GROQ_API_KEY,
)

In [ ]:
# Define Agent State
class AgentState(TypedDict):
    user_query: str
    schema_info: str
    generated_sql: str
    sql_valid: bool
    execution_result: Optional[dict]
    error_message: str
    retry_count: int
    final_answer: str
    steps_log: List[str]  # Track what the agent did



In [ ]:
#  Define Agent Nodes
def schema_lookup(state: AgentState) -> AgentState:
    """Node 1: Analyze the query and fetch relevant schema information."""
    schema = get_full_schema()
    tables = get_table_names()

    # Get sample data for context
    samples = ""
    for table in tables:
        samples += f"\n--- {table} (sample rows) ---\n"
        samples += get_sample_data(table) + "\n"

    schema_info = f"DATABASE SCHEMA:\n{schema}\n\nSAMPLE DATA:{samples}"

    log = state.get("steps_log", [])
    log.append(f"[Schema Lookup] Fetched schema for {len(tables)} tables: {', '.join(tables)}")

    return {**state, "schema_info": schema_info, "steps_log": log}


def generate_sql(state: AgentState) -> AgentState:
    """Node 2: Generate SQL from the user's natural language query."""
    error_context = ""
    if state.get("error_message"):
        error_context = f"""

PREVIOUS ATTEMPT FAILED with this error:
{state['error_message']}

Previous SQL that failed:
{state.get('generated_sql', 'N/A')}

Fix the SQL to avoid this error. Do NOT repeat the same mistake.
"""

    messages = [
        SystemMessage(content=f"""You are an expert SQL query generator for SQLite databases.
Given a database schema and a natural language question, generate ONLY the SQL query.

RULES:
- Generate valid SQLite SQL syntax only
- Use only tables and columns that exist in the schema
- Always use proper JOINs (never implicit joins)
- Use aliases for readability
- Handle aggregations, GROUP BY, HAVING correctly
- For text matching, use LIKE with % wildcards (case-insensitive)
- Return the SQL inside a ```sql code block
- Generate ONLY SELECT statements (read-only queries)
- Do NOT generate INSERT, UPDATE, DELETE, DROP, or any write operations
- If asked to modify data, generate a SELECT that says "This is a read-only system" as a column: SELECT 'This is a read-only system. I can only answer questions about data, not modify it.' AS message;
- End every SQL query with a semicolon ;

SQLITE DATE FUNCTIONS — MUST USE THESE (SQLite does NOT support EXTRACT, DATE_FORMAT, DATE_TRUNC, or YEAR()):
- Monthly grouping: strftime('%Y-%m', date_column)
- Year extraction: strftime('%Y', date_column)
- Day extraction: strftime('%d', date_column)
- NEVER use EXTRACT(), DATE_FORMAT(), YEAR(), MONTH(), DATE_TRUNC()
- CORRECT example: SELECT strftime('%Y-%m', order_date) AS month, SUM(total_amount) AS revenue FROM orders GROUP BY month ORDER BY month;
- WRONG example: SELECT EXTRACT(YEAR FROM order_date) — THIS WILL FAIL

{state['schema_info']}
{error_context}"""),
        HumanMessage(content=f"Generate SQL for: {state['user_query']}")
    ]

    response = llm.invoke(messages)
    sql = extract_sql(response.content)

    log = state.get("steps_log", [])
    log.append(f"[SQL Generator] Generated SQL: {sql}")

    return {**state, "generated_sql": sql, "steps_log": log}


def validate_sql(state: AgentState) -> AgentState:
    """Node 3: Validate the generated SQL query using safety checks + EXPLAIN."""
    sql = state["generated_sql"]
    log = state.get("steps_log", [])
    q = _normalize_sql(sql)

    # Run safety checks (same ones execute_sql uses)
    if ";" in q:
        log.append("[Validator] BLOCKED: Multiple statements detected")
        return {**state, "sql_valid": False, "error_message": "Multiple SQL statements are not allowed.",
                "retry_count": state.get("retry_count", 0) + 1, "steps_log": log}

    if DISALLOWED_KEYWORDS.search(q):
        log.append("[Validator] BLOCKED: Write operation detected")
        return {**state, "sql_valid": False, "error_message": "Only read-only SELECT queries are allowed.",
                "retry_count": state.get("retry_count", 0) + 1, "steps_log": log}

    if not q.lower().startswith(("select", "with")):
        log.append("[Validator] BLOCKED: Not a SELECT/WITH query")
        return {**state, "sql_valid": False, "error_message": "Only SELECT/WITH queries are allowed.",
                "retry_count": state.get("retry_count", 0) + 1, "steps_log": log}

    if re.search(r"\b(full\s+outer\s+join|right\s+join)\b", q, re.I):
        log.append("[Validator] BLOCKED: Unsupported JOIN type for SQLite")
        return {**state, "sql_valid": False,
                "error_message": "SQLite does not support FULL OUTER JOIN / RIGHT JOIN. Use LEFT JOIN instead.",
                "retry_count": state.get("retry_count", 0) + 1, "steps_log": log}

    # Try EXPLAIN to check syntax without executing
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    try:
        cursor.execute(f"EXPLAIN QUERY PLAN {q}")
        conn.close()
        log.append("[Validator] SQL syntax is valid")
        return {**state, "sql_valid": True, "error_message": "", "steps_log": log}
    except Exception as e:
        conn.close()
        log.append(f"[Validator] SQL validation failed: {e}")
        return {**state, "sql_valid": False, "error_message": str(e),
                "retry_count": state.get("retry_count", 0) + 1, "steps_log": log}


def execute_query(state: AgentState) -> AgentState:
    """Node 4: Execute the validated SQL query."""
    result = execute_sql(state["generated_sql"])
    log = state.get("steps_log", [])

    if result["success"]:
        log.append(f"[Executor] Query returned {result['row_count']} rows")
    else:
        log.append(f"[Executor] Execution failed: {result['error']}")

    return {**state, "execution_result": result, "steps_log": log}


def generate_response(state: AgentState) -> AgentState:
    """Node 5: Generate a natural language response from the SQL results."""
    result = state["execution_result"]
    log = state.get("steps_log", [])

    if not result or not result["success"]:
        error = result["error"] if result else "Unknown error"
        log.append(f"[Response] Failed to generate response: {error}")
        return {
            **state,
            "final_answer": f"I was unable to answer your question. Error: {error}",
            "steps_log": log,
        }

    # Format results for the LLM
    columns = result["columns"]
    rows = result["rows"]

    if not rows:
        log.append("[Response] No results found")
        return {
            **state,
            "final_answer": "The query returned no results. The data you're looking for might not exist in the database.",
            "steps_log": log,
        }

    # Limit display rows to avoid token overflow
    display_rows = rows[:50]
    result_text = f"Columns: {columns}\n"
    for row in display_rows:
        result_text += f"{row}\n"
    if len(rows) > 50:
        result_text += f"\n... and {len(rows) - 50} more rows"

    messages = [
        SystemMessage(content="""You are a helpful data analyst. Given a user's question, the SQL query used,
and the query results, provide a clear and concise natural language answer.

RULES:
- Summarize the key findings
- Mention specific numbers and values from the results
- If there are many rows, highlight the key patterns
- Format numbers nicely (e.g., currency with $, percentages with %)
- Use bullet points or tables for multiple results
- Be conversational but precise
- NEVER suggest or provide INSERT, UPDATE, DELETE, DROP, or any write SQL statements
- If the user asked to modify/delete data, remind them this is a read-only system"""),
        HumanMessage(content=f"""User Question: {state['user_query']}

SQL Query Used:
```sql
{state['generated_sql']}
```

Query Results:
{result_text}

Total rows returned: {len(rows)}

Provide a clear answer:""")
    ]

    response = llm.invoke(messages)
    log.append("[Response] Generated natural language answer")

    return {**state, "final_answer": response.content, "steps_log": log}


def handle_error(state: AgentState) -> AgentState:
    """Node 6: Handle cases where max retries are exceeded."""
    log = state.get("steps_log", [])
    log.append(f"[Error Handler] Max retries ({state['retry_count']}) exceeded")

    return {
        **state,
        "final_answer": (
            f"I apologize, but I couldn't generate a valid SQL query after {state['retry_count']} attempts.\n\n"
            f"**Your question:** {state['user_query']}\n\n"
            f"**Last error:** {state.get('error_message', 'Unknown')}\n\n"
            f"**Last SQL attempted:**\n```sql\n{state.get('generated_sql', 'N/A')}\n```\n\n"
            "Please try rephrasing your question or being more specific about the tables and columns."
        ),
        "steps_log": log,
    }


In [ ]:
# Define Routing Functions
def route_after_validation(state: AgentState) -> str:
    """Route based on SQL validation result."""
    if state["sql_valid"]:
        return "execute_query"
    elif state.get("retry_count", 0) >= 3:
        return "handle_error"
    else:
        return "generate_sql"  # Retry with error context


In [ ]:
# Build the LangGraph
def build_agent():
    """Build and compile the Text-to-SQL agent graph."""
    workflow = StateGraph(AgentState)

    # Add nodes
    workflow.add_node("schema_lookup", schema_lookup)
    workflow.add_node("generate_sql", generate_sql)
    workflow.add_node("validate_sql", validate_sql)
    workflow.add_node("execute_query", execute_query)
    workflow.add_node("generate_response", generate_response)
    workflow.add_node("handle_error", handle_error)

    # Add edges
    workflow.add_edge(START, "schema_lookup")
    workflow.add_edge("schema_lookup", "generate_sql")
    workflow.add_edge("generate_sql", "validate_sql")

    # Conditional routing after validation
    workflow.add_conditional_edges(
        "validate_sql",
        route_after_validation,
        {
            "execute_query": "execute_query",
            "generate_sql": "generate_sql",  # Retry loop
            "handle_error": "handle_error",
        },
    )

    workflow.add_edge("execute_query", "generate_response")
    workflow.add_edge("generate_response", END)
    workflow.add_edge("handle_error", END)

    return workflow.compile()


# Build the agent
agent = build_agent()
print("Agent graph built successfully!")



Agent graph built successfully!


In [ ]:
# Build the agent
agent = build_agent()
print("Agent graph built successfully!")


Agent graph built successfully!


In [ ]:
# Test the Agent
def run_agent(query: str) -> tuple:
    """Run the agent and return the answer + debug info."""
    initial_state: AgentState = {
        "user_query": query,
        "schema_info": "",
        "generated_sql": "",
        "sql_valid": False,
        "execution_result": None,
        "error_message": "",
        "retry_count": 0,
        "final_answer": "",
        "steps_log": [],
    }

    final_state = agent.invoke(initial_state)

    # Build debug info
    steps = "\n".join(final_state.get("steps_log", []))
    sql = final_state.get("generated_sql", "N/A")
    result = final_state.get("execution_result", {})

    # Format result table
    result_table = ""
    if result and result.get("success") and result.get("rows"):
        columns = result["columns"]
        rows = result["rows"]
        # Create a simple table
        result_table = " | ".join(columns) + "\n"
        result_table += " | ".join(["---"] * len(columns)) + "\n"
        for row in rows[:30]:
            result_table += " | ".join(str(v) for v in row) + "\n"
        if len(rows) > 30:
            result_table += f"\n... ({len(rows) - 30} more rows)\n"

    answer = final_state.get("final_answer", "No answer generated.")

    return answer, sql, result_table, steps




In [ ]:
# Quick test
print("\n--- Quick Test ---")
answer, sql, table, steps = run_agent("What are the top 5 most expensive products?")
print(f"SQL: {sql}")
print(f"Answer: {answer}")


--- Quick Test ---
SQL: SELECT 
    product_name, 
    price 
FROM 
    products 
ORDER BY 
    price DESC 
LIMIT 5;
Answer: Based on the query results, here are the top 5 most expensive products:

**Top 5 Most Expensive Products:**

| Rank | Product Name | Price |
| --- | --- | --- |
| 1 | Laptop Pro 15 | $1,299.99 |
| 2 | Standing Desk | $599.99 |
| 3 | Monitor 27 inch | $449.99 |
| 4 | Ergonomic Chair | $399.99 |
| 5 | Noise-Cancel Headphones | $249.99 |

The Laptop Pro 15 is the most expensive product, with a price of $1,299.99, which is more than twice the price of the second most expensive product, the Standing Desk. The top 5 products have a price range of $249.99 to $1,299.99.


In [ ]:

# Gradio UI

def gradio_chat(message: str, history: list) -> str:
    """Gradio chat handler — keeps conversation history visible."""
    if not message.strip():
        return "Please enter a question."

    answer, sql, result_table, steps = run_agent(message)

    # Build a rich response with SQL + answer
    response = f"{answer}\n\n"
    response += f"---\n**SQL Generated:**\n```sql\n{sql}\n```\n"
    if result_table:
        response += f"\n**Raw Results:**\n{result_table}\n"
    response += f"\n<details><summary>Agent Steps</summary>\n\n```\n{steps}\n```\n</details>"

    return response


# Sample questions for the UI
SAMPLE_QUESTIONS = [
    "What are the top 5 most expensive products?",
    "How many orders were placed by each customer?",
    "What is the total revenue by product category?",
    "Show me all employees in the Engineering department with salary above 100000",
    "Which customers have placed more than 2 orders?",
    "What is the average order value by month?",
    "List all products that are low in stock (less than 200 units)",
    "Who are the top 3 customers by total spending?",
    "Show the number of employees in each department",
    "What are the most popular products by number of orders?",
]

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="🤖 Agentic Text-to-SQL System",
    description=(
        "**Ask questions about your database in plain English**\n\n"
        "**Powered by:** LangGraph | Groq (Llama 3.1 8B) | SQLite\n\n"
        "**Database:** Sample company DB with departments, employees, customers, products, orders & order items."
    ),
    examples=SAMPLE_QUESTIONS,
    theme=gr.themes.Soft(),
)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


In [ ]:
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fc3bb954f22188abd3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
